# YOGA Chatbot - Full NLU Pipeline
**Yogyakarta Tourism Guide Assistant**

## Architecture:
- **NLU**: Intent Classification (10 intents) + Entity Extraction (78 kecamatan + 5 kabupaten)
- **Action Handler**: Business logic untuk setiap intent
- **Database**: 135 tempat wisata di DIY
- **NLG**: Natural response generation

## Pipeline:
```
User Input → NLU (Intent + Entity) → Action Handler → Database Filter → NLG Response
```

## 1. Install Dependencies

In [ ]:
# Install required packages
!pip install pandas numpy scikit-learn Sastrawi -q

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import json
import random
import pickle
import re
import string
import warnings
warnings.filterwarnings('ignore')

# NLP
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix

print("✅ Libraries imported successfully!")

## 3. Load Kecamatan Data

In [ ]:
# 78 Kecamatan di DIY
KECAMATAN_DIY = {
    'bantul': ['bambang lipuro', 'banguntapan', 'bantul', 'dlingo', 'imogiri',
               'jetis', 'kasihan', 'kretek', 'pajangan', 'pandak',
               'piyungan', 'pleret', 'pundong', 'sanden', 'sedayu',
               'sewon', 'srandakan'],
    'sleman': ['berbah', 'cangkringan', 'depok', 'gamping', 'godean',
               'kalasan', 'minggir', 'mlati', 'moyudan', 'ngaglik',
               'ngemplak', 'pakem', 'prambanan', 'seyegan', 'sleman',
               'tempel', 'turi'],
    'gunungkidul': ['gedangsari', 'girisubo', 'karangmojo', 'ngawen', 'nglipar',
                    'paliyan', 'panggang', 'patuk', 'playen', 'ponjong',
                    'purwosari', 'rongkop', 'sapto sari', 'semanu', 'semin',
                    'tanjungsari', 'tepus', 'wonosari'],
    'kulonprogo': ['galur', 'girimulyo', 'kalibawang', 'kokap', 'lendah',
                   'nanggulan', 'panjatan', 'pengasih', 'samigaluh', 'sentolo',
                   'temon', 'wates'],
    'yogyakarta': ['danurejan', 'gedongtengen', 'gondokusuman', 'gondomanan', 'jetis',
                   'kotagede', 'kraton', 'mantrijeron', 'mergangsan', 'ngampilan',
                   'pakualaman', 'tegalrejo', 'umbulharjo', 'wirobrajan']
}

# Flatten list
ALL_KECAMATAN = []
KECAMATAN_TO_KABUPATEN = {}

for kabupaten, kecamatan_list in KECAMATAN_DIY.items():
    for kecamatan in kecamatan_list:
        ALL_KECAMATAN.append(kecamatan)
        KECAMATAN_TO_KABUPATEN[kecamatan] = kabupaten

print(f"✅ Loaded {len(ALL_KECAMATAN)} kecamatan di DIY")

## 4. Generate Balanced Dataset (3,000 samples)

In [ ]:
print("Generating balanced dataset...")

TARGET = 300  # samples per intent
dataset = []

# 1. GREETING
greeting_base = ["halo", "hai", "hi", "hello", "hey", "selamat pagi", "selamat siang", 
                 "selamat sore", "selamat malam", "pagi", "siang", "sore", "malam",
                 "halo yoga", "hai bot", "permisi", "ada", "assalamualaikum"]
for p in greeting_base:
    dataset.append({"text": p, "intent": "greeting"})
for base in ["halo", "hai", "hi", "hey", "pagi", "siang"]:
    for suffix in ["yoga", "kak", "bot", "bro"]:
        dataset.append({"text": f"{base} {suffix}", "intent": "greeting"})
while len([d for d in dataset if d['intent'] == 'greeting']) < TARGET:
    dataset.append({"text": random.choice(["halo", "hai", "hi"]), "intent": "greeting"})

# 2. GOODBYE
goodbye_base = ["bye", "dadah", "goodbye", "sampai jumpa", "terima kasih", "thanks", 
                "makasih udahan", "oke cukup", "udah dulu", "pamit dulu"]
for p in goodbye_base:
    dataset.append({"text": p, "intent": "goodbye"})
while len([d for d in dataset if d['intent'] == 'goodbye']) < TARGET:
    dataset.append({"text": random.choice(["bye", "dadah", "makasih"]), "intent": "goodbye"})

# 3. THANKS
thanks_base = ["makasih", "thanks", "thank you", "terima kasih", "tengkyu", "tq", 
               "makasi", "trimakasih", "thanks a lot", "sip makasih"]
for p in thanks_base:
    dataset.append({"text": p, "intent": "thanks"})
while len([d for d in dataset if d['intent'] == 'thanks']) < TARGET:
    dataset.append({"text": random.choice(["makasih", "thanks"]), "intent": "thanks"})

# 4. HELP
help_base = ["help", "bantuan", "tolong", "bantu", "cara pake bot ini gimana", 
             "gimana cara pakainya", "bisa apa aja", "aku bingung", "panduan"]
for p in help_base:
    dataset.append({"text": p, "intent": "help"})
while len([d for d in dataset if d['intent'] == 'help']) < TARGET:
    dataset.append({"text": random.choice(["help", "bantuan", "gimana"]), "intent": "help"})

# 5. FALLBACK
fallback_base = ["???", "hah", "apa", "ga jelas", "ga ngerti", "bingung", "asdfghjkl"]
for p in fallback_base:
    dataset.append({"text": p, "intent": "fallback"})
while len([d for d in dataset if d['intent'] == 'fallback']) < TARGET:
    dataset.append({"text": "???", "intent": "fallback"})

# 6. TANYA_LOKASI
sample_places = ["mlati", "godean", "bantul", "sleman", "depok", "sewon", "kalasan"]
templates = ["dimana", "di mana", "termasuk kabupaten apa", "itu wilayah apa"]
for place in sample_places:
    for template in templates:
        dataset.append({"text": f"{place} {template}", "intent": "tanya_lokasi"})
while len([d for d in dataset if d['intent'] == 'tanya_lokasi']) < TARGET:
    place = random.choice(sample_places)
    template = random.choice(templates)
    dataset.append({"text": f"{place} {template}", "intent": "tanya_lokasi"})

# 7. DETAIL_WILAYAH
templates = ["info", "tentang", "jelasin", "ceritain"]
for place in sample_places:
    for template in templates:
        dataset.append({"text": f"{template} {place}", "intent": "detail_wilayah"})
while len([d for d in dataset if d['intent'] == 'detail_wilayah']) < TARGET:
    place = random.choice(sample_places)
    template = random.choice(templates)
    dataset.append({"text": f"{template} {place}", "intent": "detail_wilayah"})

# 8. REKOMENDASI_WISATA_KECAMATAN
templates = ["wisata", "rekomendasi", "tempat wisata", "wisata di"]
for place in sample_places:
    for template in templates:
        dataset.append({"text": f"{template} {place}", "intent": "rekomendasi_wisata_kecamatan"})
while len([d for d in dataset if d['intent'] == 'rekomendasi_wisata_kecamatan']) < TARGET:
    place = random.choice(sample_places)
    template = random.choice(templates)
    dataset.append({"text": f"{template} {place}", "intent": "rekomendasi_wisata_kecamatan"})

# 9. REKOMENDASI_WISATA_KABUPATEN
kabupaten_list = ["bantul", "sleman", "gunungkidul", "kulonprogo", "yogyakarta"]
templates = ["wisata", "rekomendasi", "tempat wisata", "liburan di"]
for kab in kabupaten_list:
    for template in templates:
        dataset.append({"text": f"{template} {kab}", "intent": "rekomendasi_wisata_kabupaten"})
while len([d for d in dataset if d['intent'] == 'rekomendasi_wisata_kabupaten']) < TARGET:
    kab = random.choice(kabupaten_list)
    template = random.choice(templates)
    dataset.append({"text": f"{template} {kab}", "intent": "rekomendasi_wisata_kabupaten"})

# 10. REKOMENDASI_WISATA_PROVINSI
provinsi_names = ["yogyakarta", "jogja", "diy", "yogya"]
templates = ["wisata", "rekomendasi", "tempat wisata", "liburan"]
for prov in provinsi_names:
    for template in templates:
        dataset.append({"text": f"{template} {prov}", "intent": "rekomendasi_wisata_provinsi"})
while len([d for d in dataset if d['intent'] == 'rekomendasi_wisata_provinsi']) < TARGET:
    prov = random.choice(provinsi_names)
    template = random.choice(templates)
    dataset.append({"text": f"{template} {prov}", "intent": "rekomendasi_wisata_provinsi"})

# Shuffle and create DataFrame
random.shuffle(dataset)
df_intent = pd.DataFrame(dataset)

print(f"✅ Generated {len(df_intent)} samples")
print(f"\nIntent distribution:")
print(df_intent['intent'].value_counts())
print(f"\nBalance ratio: {df_intent['intent'].value_counts().min() / df_intent['intent'].value_counts().max():.2f}")

## 5. Preprocessing & Train-Test Split

In [ ]:
print("Preprocessing text...")

# Initialize preprocessing tools
stemmer = StemmerFactory().create_stemmer()
stopwords = set(StopWordRemoverFactory().get_stop_words())

def clean(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.strip()

def preprocess(text):
    text = clean(text)
    text = ' '.join([w for w in text.split() if w not in stopwords])
    return stemmer.stem(text)

df_intent['processed'] = df_intent['text'].apply(preprocess)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    df_intent['processed'], df_intent['intent'],
    test_size=0.2, random_state=42, stratify=df_intent['intent']
)

print(f"✅ Train: {len(X_train)} samples")
print(f"✅ Test: {len(X_test)} samples")

## 6. TF-IDF Vectorization

In [ ]:
print("Creating TF-IDF features...")

tfidf = TfidfVectorizer(
    max_features=300,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.8,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"✅ TF-IDF shape: {X_train_tfidf.shape}")
print(f"✅ Vocabulary size: {len(tfidf.vocabulary_)}")

## 7. Train SVM Model

In [ ]:
print("Training SVM model...")

svm_model = SVC(
    kernel='linear',
    C=0.5,
    probability=True,
    random_state=42,
    class_weight='balanced'
)

svm_model.fit(X_train_tfidf, y_train)

print("✅ Training completed!")

## 8. Evaluate Model

In [ ]:
print("Evaluating model...\n")

# Predictions
y_pred_train = svm_model.predict(X_train_tfidf)
y_pred_test = svm_model.predict(X_test_tfidf)

# Metrics
train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_test)
train_f1 = f1_score(y_train, y_pred_train, average='macro')
test_f1 = f1_score(y_test, y_pred_test, average='macro')

print("="*70)
print("MODEL PERFORMANCE")
print("="*70)
print(f"Training Accuracy:   {train_acc:.4f} ({train_acc*100:.2f}%)")
print(f"Test Accuracy:       {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Training F1-Score:   {train_f1:.4f}")
print(f"Test F1-Score:       {test_f1:.4f}")
print(f"\nOverfitting Gap:     {(train_acc - test_acc)*100:.2f}%")

if train_acc - test_acc < 0.05:
    print("✅ Good! Model is NOT overfitting (gap < 5%)")
elif train_acc - test_acc < 0.10:
    print("⚠️  Slight overfitting (gap 5-10%)")
else:
    print("❌ Severe overfitting (gap > 10%)")

print(f"\n{'='*70}")
print("Per-Class Performance (Test Set):")
print(f"{'='*70}")
print(classification_report(y_test, y_pred_test))

## 9. Load Wisata Dataset

In [ ]:
# If running in Colab, upload wisata_yogyakarta.json first
# from google.colab import files
# uploaded = files.upload()

# For local
df_wisata = pd.read_json('data/wisata_yogyakarta.json')

print(f"✅ Loaded {len(df_wisata)} tempat wisata")
print(f"\nSample data:")
df_wisata.head()

## 10. Entity Extractor Class

In [ ]:
class EntityExtractor:
    def __init__(self):
        self.all_kecamatan = ALL_KECAMATAN
        self.kecamatan_to_kabupaten = KECAMATAN_TO_KABUPATEN
        
        self.kabupaten_keywords = {
            'bantul': ['bantul', 'kab bantul', 'kabupaten bantul'],
            'sleman': ['sleman', 'kab sleman', 'kabupaten sleman'],
            'gunungkidul': ['gunung kidul', 'gunungkidul', 'wonosari'],
            'kulonprogo': ['kulon progo', 'kulonprogo', 'wates'],
            'yogyakarta': ['kota jogja', 'kota yogyakarta', 'kota yogya']
        }
        
        self.provinsi_keywords = ['diy', 'yogyakarta', 'jogja', 'yogya']
    
    def extract(self, text):
        text_lower = text.lower().strip()
        
        # Check kecamatan
        for kecamatan in self.all_kecamatan:
            if kecamatan in text_lower:
                return {
                    "wilayah": kecamatan,
                    "tipe": "kecamatan",
                    "kabupaten": self.kecamatan_to_kabupaten.get(kecamatan)
                }
        
        # Check kabupaten
        for kabupaten, keywords in self.kabupaten_keywords.items():
            for keyword in keywords:
                if keyword in text_lower:
                    return {
                        "wilayah": kabupaten,
                        "tipe": "kabupaten",
                        "kabupaten": kabupaten
                    }
        
        # Check provinsi
        for prov_keyword in self.provinsi_keywords:
            if prov_keyword in text_lower:
                return {
                    "wilayah": "diy",
                    "tipe": "provinsi",
                    "kabupaten": None
                }
        
        return {"wilayah": None, "tipe": None, "kabupaten": None}

entity_extractor = EntityExtractor()
print("✅ Entity Extractor initialized")

## 11. NLU Pipeline Class

In [ ]:
class NLUPipeline:
    def __init__(self, svm_model, tfidf, entity_extractor, stemmer, stopwords):
        self.svm_model = svm_model
        self.tfidf = tfidf
        self.entity_extractor = entity_extractor
        self.stemmer = stemmer
        self.stopwords = stopwords
    
    def clean(self, text):
        text = text.lower()
        text = re.sub(r'\d+', '', text)
        text = text.translate(str.maketrans('', '', string.punctuation))
        return text.strip()
    
    def preprocess(self, text):
        text = self.clean(text)
        text = ' '.join([w for w in text.split() if w not in self.stopwords])
        return self.stemmer.stem(text)
    
    def predict_intent(self, text):
        processed = self.preprocess(text)
        X = self.tfidf.transform([processed])
        intent = self.svm_model.predict(X)[0]
        confidence = self.svm_model.predict_proba(X)[0].max()
        return intent, confidence
    
    def understand(self, text):
        # Extract entity
        entity = self.entity_extractor.extract(text)
        
        # Predict intent
        intent, confidence = self.predict_intent(text)
        
        # Map intent based on entity type
        if intent.startswith('rekomendasi_wisata'):
            if entity['tipe'] == 'kecamatan':
                intent = 'rekomendasi_wisata_kecamatan'
            elif entity['tipe'] == 'kabupaten':
                intent = 'rekomendasi_wisata_kabupaten'
            elif entity['tipe'] == 'provinsi':
                intent = 'rekomendasi_wisata_provinsi'
        
        return {
            "intent": intent,
            "confidence": confidence,
            "entity": entity,
            "original_text": text
        }

nlu = NLUPipeline(svm_model, tfidf, entity_extractor, stemmer, stopwords)
print("✅ NLU Pipeline initialized")

## 12. Action Handlers

In [ ]:
def action_greeting():
    responses = [
        "Halo! Ada yang bisa YOGA bantu?",
        "Hai! Mau cari rekomendasi wisata di Yogyakarta?",
        "Hello! YOGA siap membantu mencari tempat wisata di Yogyakarta!",
    ]
    return random.choice(responses)

def action_goodbye():
    responses = [
        "Terima kasih sudah menggunakan YOGA! Selamat berwisata!",
        "Sampai jumpa! Semoga liburannya menyenangkan!",
        "Bye! Jangan lupa mampir ke tempat wisata yang YOGA rekomendasikan ya!",
    ]
    return random.choice(responses)

def action_thanks():
    return "Sama-sama! Ada yang bisa dibantu lagi?"

def action_help():
    return """YOGA Help - Cara Pakai:

YOGA bisa membantu kamu untuk:
[1] Rekomendasi Wisata: "wisata mlati", "rekomendasi bantul"
[2] Tanya Lokasi: "mlati itu dimana?"
[3] Info Wilayah: "info tentang mlati"

Silakan tanya apa saja tentang wisata Yogyakarta!"""

def action_fallback():
    return "Hmm, aku kurang paham. Coba tanya: 'wisata jogja' atau 'help'"

def action_tanya_lokasi(entity):
    if not entity['wilayah']:
        return "Mau tanya lokasi mana? Sebutin nama kecamatannya ya!"
    
    wilayah = entity['wilayah']
    if entity['tipe'] == 'kecamatan':
        kabupaten = entity['kabupaten']
        return f"{wilayah.title()} adalah kecamatan di Kabupaten {kabupaten.title()}, DIY.\n\nMau wisata di {wilayah.title()}? Ketik: 'wisata {wilayah}'"
    else:
        return f"{wilayah.title()} adalah salah satu wilayah di DIY."

def action_detail_wilayah(entity):
    if not entity['wilayah']:
        return "Mau info tentang daerah mana?"
    
    wilayah = entity['wilayah']
    return f"Info tentang {wilayah.title()}:\n\n{wilayah.title()} terletak di DIY.\n\nUntuk rekomendasi wisata, ketik: 'wisata {wilayah}'"

def action_rekomendasi_wisata(entity):
    wilayah = entity['wilayah']
    tipe = entity['tipe']
    
    filtered = df_wisata.copy()
    
    if tipe == 'kecamatan' or tipe == 'kabupaten':
        kabupaten = entity['kabupaten'] if tipe == 'kabupaten' else entity.get('kabupaten')
        
        kabupaten_keywords = {
            'bantul': 'Bantul',
            'sleman': 'Sleman',
            'gunungkidul': 'Gunung Kidul',
            'kulonprogo': 'Kulon Progo',
            'yogyakarta': 'Kota Yogyakarta'
        }
        
        if kabupaten:
            kab_name = kabupaten_keywords.get(kabupaten, kabupaten.title())
            filtered = filtered[filtered['address'].str.contains(kab_name, case=False, na=False)]
        
        location_display = f"Kabupaten {kabupaten.title()}" if tipe == 'kabupaten' else f"Kecamatan {wilayah.title()}"
    else:
        location_display = "DIY (Daerah Istimewa Yogyakarta)"
    
    filtered = filtered[filtered['rating'] > 0]
    filtered = filtered.sort_values(['rating', 'total_ratings'], ascending=[False, False])
    top3 = filtered.head(3)
    
    if len(top3) == 0:
        return f"Maaf, belum ada data wisata untuk {wilayah.title() if wilayah else 'wilayah tersebut'}."
    
    response = f"Berikut rekomendasi wisata di {location_display}:\n\n"
    for i, (_, row) in enumerate(top3.iterrows(), 1):
        response += f"✨ {row['name']}\n"
        response += f"   Rating: {row['rating']}/5 ({row['total_ratings']:,} reviews)\n"
        response += f"   Lokasi: {row['address'][:60]}...\n\n"
    
    return response

print("✅ Action handlers defined")

## 13. Main Chatbot Function

In [ ]:
def process_message(user_input):
    """Main NLU → Action → NLG pipeline"""
    
    # Step 1: NLU
    nlu_result = nlu.understand(user_input)
    intent = nlu_result['intent']
    entity = nlu_result['entity']
    
    # Step 2: Action Handler
    if intent == 'greeting':
        response = action_greeting()
    elif intent == 'goodbye':
        response = action_goodbye()
    elif intent == 'thanks':
        response = action_thanks()
    elif intent == 'help':
        response = action_help()
    elif intent == 'fallback':
        response = action_fallback()
    elif intent == 'tanya_lokasi':
        response = action_tanya_lokasi(entity)
    elif intent == 'detail_wilayah':
        response = action_detail_wilayah(entity)
    elif intent in ['rekomendasi_wisata_kecamatan', 'rekomendasi_wisata_kabupaten', 'rekomendasi_wisata_provinsi']:
        response = action_rekomendasi_wisata(entity)
    else:
        response = action_fallback()
    
    return response

print("✅ Chatbot ready!")

## 14. Test Chatbot

In [ ]:
print("="*70)
print("TESTING YOGA CHATBOT")
print("="*70)
print()

test_inputs = [
    "halo yoga",
    "wisata mlati",
    "rekomendasi bantul",
    "mlati itu dimana?",
    "wisata jogja",
    "makasih ya",
    "bye"
]

for user_input in test_inputs:
    response = process_message(user_input)
    print(f"You: {user_input}")
    print(f"YOGA: {response}")
    print()

## 15. Interactive Chat (Optional)

In [ ]:
print("="*70)
print("YOGA CHATBOT - Interactive Mode")
print("Type 'exit' to quit")
print("="*70)
print()

while True:
    user_input = input("You: ").strip()
    
    if not user_input:
        continue
    
    if user_input.lower() in ['exit', 'quit', 'keluar']:
        print("\nYOGA: Terima kasih sudah menggunakan YOGA! Selamat berwisata! 👋")
        break
    
    response = process_message(user_input)
    print(f"\nYOGA: {response}\n")